# Performance
Evaluates the performance/alignment (proportional match) of LLMs' generated distractors with human annotated ones (from Eedi).

In [ ]:
import os
import json
from tqdm import tqdm
from dotenv import load_dotenv

import pandas as pd
from openai import OpenAI

from src.datasets import ADataset, get_or_create_dataset
from src.equality import SemanticEqualityChecker, EqualityChecker
from src.evaluation import number_correct, number_repetitions, proportional_match, partial_match, exact_match, get_mean_and_ci
from src.model_configurations import gpt_4_1_mini_det_config

load_dotenv()

### Datasets

In [ ]:
equivalence_check_path = "cache/semantic_equivalence_checker.pkl"

In [ ]:
eedi_dataset = get_or_create_dataset("eedi_data", n_limit=500)
print(f"We have {len(eedi_dataset)} EEDI questions")

datasets_by_datafolder = {
    "eedi_data": eedi_dataset
}

equality_model_config = gpt_4_1_mini_det_config
equality_client = OpenAI(base_url=equality_model_config["base_url"], api_key=os.environ.get(equality_model_config["api_key_var"], None))
if os.path.exists(equivalence_check_path):
    print("Loading existing semantic equality checker")
    semantic_equality_checker = SemanticEqualityChecker.load(equality_client, equivalence_check_path)
else:
    semantic_equality_checker = SemanticEqualityChecker(equality_client, equality_model_config)
    semantic_equality_checker.save(equivalence_check_path)

### Evaluation

In [ ]:
def analyze_responses(dataset: ADataset, responses_by_datapointid: dict, results_by_datapointid: dict, equality_checker: EqualityChecker):
    problem_by_datapointid = {
        str(dpid): dataset[dpid]["Problem"]["Question"]
        for dpid in range(len(dataset))
    }
    gt_distractors_by_datapointid = { 
        str(dpid): dataset[dpid]["Choices"]["Distractors"]
        for dpid in range(len(dataset))
    }
    gt_correct_by_datapointid = { 
        str(dpid): dataset[dpid]["Choices"]["CorrectAnswer"] 
        for dpid in range(len(dataset))
    }
    num_cor_sol_steps_by_datapointid = {
        str(dpid): dataset[dpid]["Problem"]["NumReasoningSteps"]
        for dpid in range(len(dataset))
    }

    for dpid in tqdm(responses_by_datapointid.keys()):
        dpid = str(dpid)
        if dpid in results_by_datapointid: continue

        responses_of_datapointid = responses_by_datapointid[dpid]
        problem = problem_by_datapointid[dpid]
        responses = [v for k,v in responses_of_datapointid.items() if k.endswith("_answer")]
        groundtruths = gt_distractors_by_datapointid[dpid]
        correct_answer = gt_correct_by_datapointid[dpid]

        results_by_datapointid[dpid] = {
            "proportional_match": proportional_match(equality_checker, problem, responses, groundtruths),
            "exact_match": exact_match(equality_checker, problem, responses, groundtruths),
            "partial_match": partial_match(equality_checker, problem, responses, groundtruths),
            "distractors": responses,
            "num_distractors": len(responses),
            "repetitions": responses_of_datapointid.get("statistics", {}).get("repetitions", number_repetitions(equality_checker, problem, responses)),
            "number_correct": responses_of_datapointid.get("statistics", {}).get("number_correct", number_correct(equality_checker, problem, responses, correct_answer)),
            "attempts": responses_of_datapointid.get("statistics", {}).get("attempts", 1),
            "problem_len_chars": len(problem),
            "num_cor_sol_steps_by_datapointid": num_cor_sol_steps_by_datapointid[dpid]
        }

    return pd.DataFrame([{"Id": dpid, **result} for dpid,result in results_by_datapointid.items()])    

In [ ]:
for data_folder,dataset in datasets_by_datafolder.items():
    results_folder = os.path.join(data_folder, "joint_results")
    
    for filename in os.listdir(results_folder):
        if not filename.endswith("_responses_by_datapointid.json"): continue
        
        print(f"Processing responses: {os.path.join(results_folder, filename)}")
        with open(os.path.join(results_folder, filename), "r+") as f:
            responses_by_datapointid = json.loads(f.read())

            results_file = os.path.join(results_folder, f"{filename.replace('_responses_by_datapointid.json', '')}_results.csv")
            
            # load existing results
            results_by_datapointid = {}
            if os.path.exists(results_file):
                results_df = pd.read_csv(results_file)
                results_by_datapointid = {
                    str(row["Id"]): {k: row[k] for k in results_df.columns if k != "Id"}
                    for _, row in results_df.iterrows()
                }

            results_df = analyze_responses(dataset, responses_by_datapointid, results_by_datapointid, semantic_equality_checker)
            results_df.to_csv(results_file, index=False)

            semantic_equality_checker.save(equivalence_check_path)

In [ ]:
def plot_results(data_folder: str, filter_unsolvable: bool = False, filter_cutoff: bool = True):
    results_by_setting = {}

    results_folder = os.path.join(data_folder, "joint_results")
    for filename in os.listdir(results_folder):
        if "oss" not in filename: continue
        
        if not filename.endswith("_responses_by_datapointid.json"): continue
        setting = filename.replace("_responses_by_datapointid.json", "")
        try: 
            results_by_setting[setting] = pd.read_csv(os.path.join(results_folder, f"{setting}_results.csv"))
        except Exception as e:
            print(f"Failed to load results for {setting}")

    if filter_unsolvable:
        results_by_setting = {
            setting: df[df["Id"].apply(lambda x: eedi_dataset[int(x)]["Problem"]["Solvable"])]
            for setting,df in results_by_setting.items()
        }

    if filter_cutoff:
        results_by_setting = {
            setting: df[df["Id"].apply(lambda x: eedi_dataset[int(x)]["Problem"]["Solvable"])]
            for setting,df in results_by_setting.items()
        }


    for setting,result_df in results_by_setting.items():
        print(f"[{setting}]")

        for c in result_df.columns:
            if c == "Id" or c == "distractors": continue
            mean,confidence = get_mean_and_ci(result_df[c])
            print(f"\t{c}: {mean:.3f} ± {confidence:.3f}")

In [ ]:
plot_results("eedi_data", filter_unsolvable=True, filter_cutoff=False)